<a href="https://colab.research.google.com/github/Sakshamag21/github_colab_link/blob/main/Copy_of_whisper_hindi_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers librosa hindi-xlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 7.5 MB/s eta 0:00:00


In [ ]:
!pip install ai4bharat-transliteration

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 22.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl (from fairseq->ai4bharat-transliteration) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/e5/f6/043b6d255dd6fbf2025110cea35b87f4c5100a181681d8eab496269f0d5b/omegaconf-2.0.5-py3-none-any.whl (from fairseq->ai4bharat-transliteration) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need 

In [ ]:
!pip install indic-transliteration unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.0 MB/s eta 0:00:00


## Local Inference on GPU
Model page: https://huggingface.co/vasista22/whisper-hindi-small

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vasista22/whisper-hindi-small)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="vasista22/whisper-hindi-small")

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained("vasista22/whisper-hindi-small")
model = AutoModelForSpeechSeq2Seq.from_pretrained("vasista22/whisper-hindi-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

In [ ]:
import torch
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# --------- CONFIG ---------
audio_path = "test_audio.wav"
# model_id = "vasista22/whisper-hindi-small"
model_id = "openai/whisper-medium"
# model_id = "openai/whisper-large-v3"
chunk_length_s = 10  # seconds per chunk

# --------- LOAD MODEL ---------
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)
# model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
#     language="hi",
#     task="transcribe"
# )
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# --------- LOAD AUDIO ---------
audio, sr = librosa.load(audio_path, sr=16000)

# --------- CHUNK FUNCTION ---------
def split_audio(audio, sr, chunk_length_s):
    chunk_size = int(chunk_length_s * sr)
    total_length = len(audio)

    chunks = []
    for start in range(0, total_length, chunk_size):
        end = start + chunk_size
        chunk = audio[start:end]
        chunks.append(chunk)

    return chunks

chunks = split_audio(audio, sr, chunk_length_s)

# --------- TRANSCRIBE CHUNKS ---------
full_transcription = ""
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}...")

    inputs = processor(
        chunk,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(device)

    # ✅ Set Hindi decoding properly
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="hi",
        task="transcribe"
    )

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    full_transcription += text + " "

    print(text)

# --------- OUTPUT ---------
print("\nFinal Transcription:\n")
print(full_transcription.strip())

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Processing chunk 1/1...
 यहाँ एक लूश राओ परी रहे हैं।

Final Transcription:

यहाँ एक लूश राओ परी रहे हैं।


In [ ]:
import torch
import librosa
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# --------- CONFIG ---------
audio_path = "kirana_store_audio.m4a"
model_id = "openai/whisper-large-v3"
chunk_length_s = 30
overlap_s = 5

# --------- DEVICE ---------
device = "cuda" if torch.cuda.is_available() else "cpu"

# --------- LOAD MODEL ---------
processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

# --------- LOAD AUDIO ---------
audio, sr = librosa.load(audio_path, sr=16000)

# Normalize audio
audio = librosa.util.normalize(audio)

# --------- CHUNKING ---------
def split_audio_overlap(audio, sr, chunk_length_s=30, overlap_s=8):
    chunk_size = int(chunk_length_s * sr)
    overlap_size = int(overlap_s * sr)

    chunks = []
    start = 0

    while start < len(audio):
        end = start + chunk_size
        chunks.append(audio[start:end])
        start += (chunk_size - overlap_size)

    return chunks

chunks = split_audio_overlap(audio, sr, chunk_length_s, overlap_s)

# --------- CLEAN DUPLICATES ---------
def clean_text(text):
    words = text.split()
    cleaned = []

    for w in words:
        if not cleaned or w != cleaned[-1]:
            cleaned.append(w)

    return " ".join(cleaned)

# --------- TRANSCRIPTION ---------
full_text = ""

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}...")

    inputs = processor(
        chunk,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(device)

    if device == "cuda":
        input_features = input_features.half()

    # --------- INFERENCE ---------
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device == "cuda")):
        # predicted_ids = model.generate(
        #     input_features,
        #     max_new_tokens=225,
        #     num_beams=5,
        #     temperature=[0.0, 0.2, 0.4],
        #     repetition_penalty=0.8,
        #     no_repeat_ngram_size=2
        # )

        predicted_ids = model.generate(
            input_features,
            language="hi",
            task="transcribe",
            temperature=0.0,
            do_sample=False,
            num_beams=1
        )

    raw_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    print("RAW:", raw_text)
    print("-" * 50)

    full_text += raw_text + " "

# --------- FINAL CLEANUP ---------
final_output = clean_text(full_text)

# --------- OUTPUT ---------
print("\n===== FINAL TRANSCRIPTION =====\n")
print(final_output)

# --------- SAVE ---------
with open(f"output_kirana_{chunk_length_s}.txt", "w", encoding="utf-8") as f:
    f.write(final_output)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

/tmp/ipykernel_3329/2864437566.py:24: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Processing chunk 1/14...


/tmp/ipykernel_3329/2864437566.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device == "cuda")):


RAW:  जैसे यह बिल होता है यह ट्रोर अपने आप ओपन हो जाती है अब यह जो एप्पल ट्रोली है इसकी एक खासियत है यह हेंडल ऐसे खुल जाता है और इसके वील्स होते हैं आप ऐसे मूव कर सकते हो तेरा भाई 40 का मोल बनकर तयार है उस पूरी वीडियो में बताऊंगा कि रैक का कितना खर्चा आया लाइट्स का कितना खर्चा आया और यहाँ पर जो बासकेट्स है उनका क्या एक्सपेंस आया मतलब टोटल खर्चा बताया जाएगा इस पूरी वीडियो में तो आप वीडियो को पूरा एंड तक देखना ठीक है स्टार्ट करते हम रैक से जैसे ह
--------------------------------------------------
Processing chunk 2/14...
RAW:  वीडियो को पूरा एंड तक देखना ठीक है स्टार्ट करते हैं रैक से जैसे हमने यहां पर वोल रैक लगाए हैं और लगभग इसकी जो हाइट रखे ना सेवन फीट रखी है इसमें बड़ा ही प्यारा लुक आता है अगर हम डार्क शेट पर मैटीरियल लगाते हैं देख सकते हैं आप कितना हाइलाइट हो रहा है मैटीरियल यहां पर सारा और रैक है ना पीछे डार्क शेट में और यह वाइट स्ट्रिप है जैसे हमारी कंपनी का नाम हमने लिखा है यहां पर ठीक उसी तरीके से यहां पर है ना प्राइस लिस्ट वगैरह भी आ जाती है जैसे �
-----------------------------

In [ ]:
!git clone https://github.com/ggerganov/whisper.cpp


Cloning into 'whisper.cpp'...
remote: Enumerating objects: 32735, done.
remote: Counting objects: 100% (504/504), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 32735 (delta 339), reused 308 (delta 308), pack-reused 32231 (from 3)
Receiving objects: 100% (32735/32735), 34.16 MiB | 21.92 MiB/s, done.
Resolving deltas: 100% (23841/23841), done.


In [ ]:
import torch
import librosa
import re

from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
from hindi_xlit import HindiTransliterator

# -------- CONFIG --------
audio_path = "kirana_store_audio.m4a"
model_id = "openai/whisper-large-v3"
chunk_length_s = 15
overlap_s = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------- LOAD WHISPER --------
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

# -------- LOAD TRANSLITERATOR --------
transliterator = HindiTransliterator()

# -------- AUDIO --------
audio, sr = librosa.load(audio_path, sr=16000)
audio = librosa.util.normalize(audio)

def split_audio_overlap(audio, sr, chunk_length_s=15, overlap_s=5):
    chunk_len = int(chunk_length_s * sr)
    overlap_len = int(overlap_s * sr)
    chunks = []
    start = 0
    while start < len(audio):
        end = start + chunk_len
        chunks.append(audio[start:end])
        start += (chunk_len - overlap_len)
    return chunks

chunks = split_audio_overlap(audio, sr, chunk_length_s, overlap_s)

# -------- CLEAN DUPLICATES --------
def clean_text(text):
    words = text.split()
    cleaned = []
    for w in words:
        if not cleaned or w != cleaned[-1]:
            cleaned.append(w)
    return " ".join(cleaned)

# -------- TRANSCRIBE & TRANSLITERATE --------
full_roman = ""
raw_full = ""

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}")

    # Whisper features
    inputs = processor(chunk, sampling_rate=16000, return_tensors="pt")
    input_feats = inputs.input_features.to(device)
    if device=="cuda":
        input_feats = input_feats.half()

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device=="cuda")):
        output_ids = model.generate(
            input_feats,
            max_new_tokens=400,
            num_beams=5
        )

    raw_text = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    raw_full += raw_text + " "

    print("WHISPER RAW:", raw_text)

    # Transliterate — hindi_xlit
    try:
        roman = transliterator.transliterate(raw_text)
    except Exception as e:
        print("Transliteration error:", e)
        roman = raw_text  # fallback

    print("ROMAN:", roman)
    print("-"*40)

    full_roman += roman + " "

# Remove duplicate overlaps
final_output = clean_text(full_roman)

print("\n===== FINAL ROMAN TRANSCRIPTION =====\n")
print(final_output)

# Save
with open(f"output_kirana_{chunk_length_s}.txt", "w", encoding="utf-8") as f:
    f.write(final_output)

with open("output_kirana_raw.txt", "w", encoding="utf-8") as f:
    f.write(raw_full)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

/tmp/ipykernel_3329/2638051686.py:28: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Processing chunk 1/34


/tmp/ipykernel_3329/2638051686.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device=="cuda")):
Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WHISPER RAW:  जैसे ये बिल होता है ये ड्रोर अपने आप ओपन हो जाती है अब ये जो एपल ड्रोली है इसकी एक खासियत है ये हेंडल ऐसे खुल जाता है और इसके वील्स होता है आप ऐसे मोव कर सकते हो तेरा बाई 40 का मोल बनकर त्यार है उस पुरी वीडियो में बताऊँगा कि
ROMAN:  जैसे ये बिल होता है ये ड्रोर अपने आप ओपन हो जाती है अब ये जो एपल ड्रोली है इसकी एक खासियत है ये हेंडल ऐसे खुल जाता है और इसके वील्स होता है आप ऐसे मोव कर सकते हो तेरा बाई 40 का मोल बनकर त्यार है उस पुरी वीडियो में बताऊँगा कि
----------------------------------------
Processing chunk 2/34


...

WHISPER RAW:  उठाकर भी रख सकते हैं ऐसे अगर आप सबको अगर आप सबको यह वीडियो पसंद आई तो वीडियो को जरूर लाइक करना चैनल को सब्सक्राइब करोगे ऐसे वीडियो आते रहेंगे शॉप की लोकेशन मैंने वीडियो के नीचे मेंशन कर दिया वहां जाकर विजिट
ROMAN:  उठाकर भी रख सकते हैं ऐसे अगर आप सबको अगर आप सबको यह वीडियो पसंद आई तो वीडियो को जरूर लाइक करना चैनल को सब्सक्राइब करोगे ऐसे वीडियो आते रहेंगे शॉप की लोकेशन मैंने वीडियो के नीचे मेंशन कर दिया वहां जाकर विजिट
----------------------------------------
Processing chunk 33/34


Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WHISPER RAW:  जो आते रहेंगे शॉप की लोकेशन मैंने वीडियो के नीचे मेंशन कर दिया वहां जाकर विजिट करके देख सकते हो थैंक यू हुआ है
ROMAN:  जो आते रहेंगे शॉप की लोकेशन मैंने वीडियो के नीचे मेंशन कर दिया वहां जाकर विजिट करके देख सकते हो थैंक यू हुआ है
----------------------------------------
Processing chunk 34/34


Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WHISPER RAW:  Редактор субтитров А.Семкин Корректор А.Егорова
ROMAN:  Редактор субтитров А.Семкин Корректор А.Егорова
----------------------------------------

===== FINAL ROMAN TRANSCRIPTION =====

जैसे ये बिल होता है ये ड्रोर अपने आप ओपन हो जाती है अब ये जो एपल ड्रोली है इसकी एक खासियत है ये हेंडल ऐसे खुल जाता है और इसके वील्स होता है आप ऐसे मोव कर सकते हो तेरा बाई 40 का मोल बनकर त्यार है उस पुरी वीडियो में बताऊँगा कि सकते हो तेरा भाई 40 का मोल बनकर तैयार है उस पूरी वीडियो में बताऊंगा कि रैक का कितना खर्चा आया लाइट्स का कितना खर्चा आया और यहां पर जो बास्केट्स है उनका क्या एक्सपेंस आया मतलब टोटल खर्चा बताया जाएगा इस पूरी वीडियो में तो क्या expense आया मतलब total कर्चा बताया जाएगा इस पूरी वीडियो में तो आप वीडियो को पूरा end तक देखना ठीक है start करते हैं हम rack से जैसे हमने यहाँ पर wall rack लगाए हैं और लगभग इसकी जो height रखी है ना 7 feet रखी है इसमें बड़ा ही प्यारा लगाए हैं और लगभग इसकी जो हाइट रखे ना 7 फीट रखी है इसमें बड़ा ही प्यारा लुक आता है अगर हम डार्क शेड पर मैटीरियल लगाते है

In [ ]:
from indic_transliteration.sanscript import transliterate, DEVANAGARI, ITRANS
import re

# -------- LOAD RAW FILE --------
with open("output_kirana_15.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# -------- HINDI WORD DETECTOR --------
def is_hindi(word):
    return re.search(r'[\u0900-\u097F]', word) is not None

# -------- WORD-BY-WORD TRANSLITERATION --------
def transliterate_word_by_word(text):
    words = text.split()
    output_words = []

    for word in words:
        try:
            if is_hindi(word):
                roman = transliterate(word, DEVANAGARI, ITRANS)
            else:
                roman = word  # keep English/numbers unchanged
        except Exception as e:
            print(f"Error on word: {word} -> {e}")
            roman = word  # fallback

        output_words.append(roman)

    return " ".join(output_words)

# -------- RUN --------
roman_output = transliterate_word_by_word(raw_text)

# -------- SAVE --------
with open("output_kirana_roman_clean_15.txt", "w", encoding="utf-8") as f:
    f.write(roman_output)

print("\n===== WORD-BY-WORD ROMAN OUTPUT =====\n")
print(roman_output[:1000])  # preview


===== WORD-BY-WORD ROMAN OUTPUT =====

jaise ye bila hotA hai ye Drora apane Apa opana ho jAtI hai aba ye jo epala DrolI hai isakI eka khAsiyata hai ye heMDala aise khula jAtA hai aura isake vIlsa hotA hai Apa aise mova kara sakate ho terA bAI 40 kA mola banakara tyAra hai usa purI vIDiyo meM batAU.NgA ki sakate ho terA bhAI 40 kA mola banakara taiyAra hai usa pUrI vIDiyo meM batAUMgA ki raika kA kitanA kharchA AyA lAiTsa kA kitanA kharchA AyA aura yahAM para jo bAskeTsa hai unakA kyA eksapeMsa AyA matalaba ToTala kharchA batAyA jAegA isa pUrI vIDiyo meM to kyA expense AyA matalaba total karchA batAyA jAegA isa pUrI vIDiyo meM to Apa vIDiyo ko pUrA end taka dekhanA ThIka hai start karate haiM hama rack se jaise hamane yahA.N para wall rack lagAe haiM aura lagabhaga isakI jo height rakhI hai nA 7 feet rakhI hai isameM ba.DA hI pyArA lagAe haiM aura lagabhaga isakI jo hAiTa rakhe nA 7 phITa rakhI hai isameM ba.DA hI pyArA luka AtA hai agara hama DArka sheDa para maiTIriyala lagAte haiM 

In [ ]:
!git clone https://github.com/ggerganov/whisper.cpp

Cloning into 'whisper.cpp'...
remote: Enumerating objects: 32752, done.
remote: Counting objects: 100% (496/496), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 32752 (delta 289), reused 256 (delta 255), pack-reused 32256 (from 4)
Receiving objects: 100% (32752/32752), 34.15 MiB | 8.63 MiB/s, done.
Resolving deltas: 100% (23837/23837), done.


In [ ]:
%cd whisper.cpp

/content/whisper.cpp


In [ ]:
!make -j

cmake -B build 
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assemb

In [ ]:
import subprocess
import librosa
import soundfile as sf
from hindi_xlit import HindiTransliterator
import os

# -------- CONFIG --------
audio_path = "kirana_store_audio.m4a"
wav_path = "temp.wav"
model_path = "models/ggml-base.bin"
whisper_cpp_path = "./main"   # path to whisper.cpp binary

chunk_length_s = 15
overlap_s = 5

# -------- TRANSLITERATOR --------
transliterator = HindiTransliterator()

# -------- CONVERT AUDIO TO WAV --------
audio, sr = librosa.load(audio_path, sr=16000)
audio = librosa.util.normalize(audio)
sf.write(wav_path, audio, 16000)

# -------- SPLIT AUDIO --------
def split_audio_overlap(audio, sr, chunk_length_s=15, overlap_s=5):
    chunk_len = int(chunk_length_s * sr)
    overlap_len = int(overlap_s * sr)
    chunks = []
    start = 0
    while start < len(audio):
        end = start + chunk_len
        chunks.append(audio[start:end])
        start += (chunk_len - overlap_len)
    return chunks

chunks = split_audio_overlap(audio, sr, chunk_length_s, overlap_s)

# -------- CLEAN DUPLICATES --------
def clean_text(text):
    words = text.split()
    cleaned = []
    for w in words:
        if not cleaned or w != cleaned[-1]:
            cleaned.append(w)
    return " ".join(cleaned)

# -------- RUN WHISPER.CPP --------
def run_whisper_cpp(wav_file):
    result = subprocess.run(
        [
            whisper_cpp_path,
            "-m", model_path,
            "-f", wav_file,
            "-nt",  # no timestamps
        ],
        capture_output=True,
        text=True
    )
    return result.stdout

# -------- PROCESS --------
full_roman = ""
raw_full = ""

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}")

    chunk_file = f"chunk_{i}.wav"
    sf.write(chunk_file, chunk, 16000)

    raw_text = run_whisper_cpp(chunk_file)
    raw_full += raw_text + " "

    print("WHISPER RAW:", raw_text)

    try:
        roman = transliterator.transliterate(raw_text)
    except Exception as e:
        print("Transliteration error:", e)
        roman = raw_text

    print("ROMAN:", roman)
    print("-"*40)

    full_roman += roman + " "

    os.remove(chunk_file)

# -------- FINAL OUTPUT --------
final_output = clean_text(full_roman)

print("\n===== FINAL ROMAN TRANSCRIPTION =====\n")
print(final_output)

# Save
with open("output_kirana_cpp.txt", "w", encoding="utf-8") as f:
    f.write(final_output)

with open("output_kirana_raw_cpp.txt", "w", encoding="utf-8") as f:
    f.write(raw_full)